# FFOR de la grille 87_0 sans et avec BelalpSolar

Executer d'abord `grid_generation_belalp.ipynb` pour reconstruire les deux grilles et exporter leurs variables.

In [ ]:
import pickle
from pathlib import Path

data_dir = Path("Data")
scenario_file = data_dir / "ffor_grid_scenarios.pkl"
if not scenario_file.exists():
    raise FileNotFoundError(
        f"Impossible de trouver {scenario_file}. "
        "Execute d'abord grid_generation_belalp.ipynb."
    )

with open(scenario_file, "rb") as f_pkl:
    scenarios = pickle.load(f_pkl)

scenario_without_belalp = scenarios["without_belalp"]
scenario_with_belalp = scenarios["with_belalp"]

for key, scenario in scenarios.items():
    print(f"{key}: {scenario['n_nodes']} bus, {len(scenario['line_data'])} lignes, P_base={scenario['P_base']:.3f} MW, Q_base={scenario['Q_base']:.3f} MVAr")

## Solveur FFOR commun aux deux scenarios

Le solveur etudie le reseau moyenne tension vu depuis le bus 19. Le power flow linearise utilise les quatre Jacobiennes analytiques `J_Ptheta`, `J_PU`, `J_Qtheta` et `J_QU`.

Les conventions sont les suivantes :

- `P_pv >= 0` : injection active d'un PV ;
- `P_hp <= 0` : consommation active d'une pompe a chaleur ;
- `Q_pv > 0` : injection reactive ; `Q_pv < 0` : absorption reactive ;
- les limites de ligne portent sur le flux OPF de base additionne a la variation linearisee.

Chaque point obtenu est ensuite rejoue dans Pandapower afin de refuser les etats qui depassent reellement une limite de tension ou de ligne.


In [ ]:
import copy
import numpy as np
import pandapower as pp
import gurobipy as gp
from gurobipy import GRB


def solve_ffor_direction(scenario, a, b):
    """Resout le FFOR MV avec le power flow linearise par les Jacobiennes."""
    nodes = scenario["nodes"]
    pcc_bus = scenario["pcc_bus"]
    bus_position = {bus: index for index, bus in enumerate(nodes)}
    model = gp.Model(f"FFOR_{scenario['name']}")
    model.Params.OutputFlag = 0

    Pk = model.addVars(nodes, lb=-GRB.INFINITY, name="Pk")
    Qk = model.addVars(nodes, lb=-GRB.INFINITY, name="Qk")
    theta = model.addVars(nodes, lb=-GRB.INFINITY, name="theta")
    delta_U = model.addVars(nodes, lb=-GRB.INFINITY, name="U")
    P_pv = model.addVars(nodes, lb=0.0, name="Ppv")
    Q_pv = model.addVars(nodes, lb=-GRB.INFINITY, name="Qpv")
    # P_hp < 0 represente une consommation. La borne inferieure par defaut
    # de Gurobi vaut 0; elle doit donc etre explicitement desactivee.
    P_hp = model.addVars(nodes, lb=-GRB.INFINITY, ub=0.0, name="Php")

    for bus in nodes:
        position = bus_position[bus]
        if bus != pcc_bus:
            model.addConstr(Pk[bus] == P_pv[bus] + P_hp[bus] + float(scenario["P_load"].get(bus, 0.0)))
            model.addConstr(Qk[bus] == Q_pv[bus] + float(scenario["Q_load"].get(bus, 0.0)))

        model.addConstr(
            Pk[bus] == scenario["P_ref"][position] + gp.quicksum(
                scenario["J_Ptheta"][position, other_position] * theta[other_bus]
                + scenario["J_PU"][position, other_position] * delta_U[other_bus]
                for other_position, other_bus in enumerate(nodes)
            )
        )
        model.addConstr(
            Qk[bus] == scenario["Q_ref"][position] + gp.quicksum(
                scenario["J_Qtheta"][position, other_position] * theta[other_bus]
                + scenario["J_QU"][position, other_position] * delta_U[other_bus]
                for other_position, other_bus in enumerate(nodes)
            )
        )

        pv_pmax = float(scenario["P_pv_available"].get(bus, 0.0))
        pv_qmax = float(scenario["Q_pv_max"].get(bus, 0.0))
        inverter_smax = float(scenario["S_inv_max"].get(bus, 0.0))
        model.addConstr(P_pv[bus] >= 0.0)
        model.addConstr(P_pv[bus] <= pv_pmax)
        model.addConstr(Q_pv[bus] <= pv_qmax)
        model.addConstr(Q_pv[bus] >= -pv_qmax)
        model.addConstr(P_hp[bus] >= float(scenario["P_hp_max"].get(bus, 0.0)))
        if inverter_smax > 0:
            model.addQConstr(P_pv[bus] * P_pv[bus] + Q_pv[bus] * Q_pv[bus] <= inverter_smax * inverter_smax)
        else:
            model.addConstr(P_pv[bus] == 0.0)
            model.addConstr(Q_pv[bus] == 0.0)

        model.addConstr(scenario["V"] + delta_U[bus] >= scenario["delta_Umin"] * scenario["V"])
        model.addConstr(scenario["V"] + delta_U[bus] <= scenario["delta_Umax"] * scenario["V"])

    for _, line in scenario["line_data"].iterrows():
        from_bus = int(line["from_bus"])
        to_bus = int(line["to_bus"])
        angle_delta = theta[from_bus] - theta[to_bus]
        voltage_delta = delta_U[from_bus] - delta_U[to_bus]
        # Limite appliquee au flux total: flux OPF de base + variation
        # linearisee coherente avec les Jacobiennes P-theta/P-U/Q-theta/Q-U.
        Pij = (
            float(line["P_base"])
            - float(line["b"]) * angle_delta
            + float(line["g"]) * voltage_delta
        )
        Qij = (
            float(line["Q_base"])
            - float(line["g"]) * angle_delta
            - float(line["b"]) * voltage_delta
        )
        model.addQConstr(Pij * Pij + Qij * Qij <= float(line["S_max"]) ** 2)

    model.addConstr(theta[pcc_bus] == 0.0)
    model.addConstr(delta_U[pcc_bus] == 0.0)
    model.setObjective(a * Pk[pcc_bus] + b * Qk[pcc_bus], GRB.MINIMIZE)
    model.optimize()

    if model.status == GRB.OPTIMAL:
        result = {
            "P_linear": Pk[pcc_bus].X,
            "Q_linear": Qk[pcc_bus].X,
            "delta_p": {
                bus: 0.0 if bus == pcc_bus else Pk[bus].X - float(scenario["P_ref"][bus_position[bus]])
                for bus in nodes
            },
            "delta_q": {
                bus: 0.0 if bus == pcc_bus else Qk[bus].X - float(scenario["Q_ref"][bus_position[bus]])
                for bus in nodes
            },
        }
    else:
        result = None
    model.dispose()
    return result


def validate_ac_solution(scenario, solution):
    """Rejoue une solution dans Pandapower et refuse les points AC non physiques."""
    net = copy.deepcopy(scenario["mv_net"])
    for bus in scenario["nodes"]:
        delta_p = float(solution["delta_p"][bus])
        delta_q = float(solution["delta_q"][bus])
        if abs(delta_p) > 1e-10 or abs(delta_q) > 1e-10:
            pp.create_sgen(net, bus=bus, p_mw=delta_p, q_mvar=delta_q, name="FFOR AC validation")
    try:
        pp.runpp(net, calculate_voltage_angles=True, numba=False, init="auto")
    except Exception:
        return {"valid": False, "reason": "power flow AC non convergent"}
    vm_pu = net.res_bus.reindex(scenario["nodes"])["vm_pu"]
    loading = net.res_line.loc[net.line["in_service"], "loading_percent"]
    voltage_ok = bool((vm_pu >= scenario["delta_Umin"] - 1e-6).all() and (vm_pu <= scenario["delta_Umax"] + 1e-6).all())
    loading_ok = bool((loading <= 100.0 + 1e-6).all())
    return {
        "valid": voltage_ok and loading_ok,
        "reason": "ok" if voltage_ok and loading_ok else "limite AC depassee",
        "P": float(net.res_ext_grid["p_mw"].sum()),
        "Q": float(net.res_ext_grid["q_mvar"].sum()),
        "v_min_pu": float(vm_pu.min()),
        "v_max_pu": float(vm_pu.max()),
        "line_loading_max_percent": float(loading.max()),
    }


def compute_ffor(scenario, n_angles=72):
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    P_points = []
    Q_points = []
    print(f"Calcul FFOR: {scenario['name']}")
    for index, phi in enumerate(angles):
        solution = solve_ffor_direction(scenario, np.cos(phi), np.sin(phi))
        if solution is not None:
            ac_result = validate_ac_solution(scenario, solution)
        else:
            ac_result = {"valid": False, "reason": "optimisation infaisable"}
        if ac_result["valid"]:
            P_points.append(ac_result["P"])
            Q_points.append(ac_result["Q"])
            print(f"  {index + 1}/{n_angles} | phi={np.degrees(phi):.1f} deg | P_AC={ac_result['P']:.3f}, Q_AC={ac_result['Q']:.3f}")
        else:
            print(f"  {index + 1}/{n_angles} | phi={np.degrees(phi):.1f} deg | point refuse: {ac_result['reason']}")
    if len(P_points) < 3:
        raise RuntimeError(f"Pas assez de points optimaux pour tracer {scenario['name']}.")
    return {"P": P_points + [P_points[0]], "Q": Q_points + [Q_points[0]]}


## Calcul et comparaison des deux FFOR

In [ ]:
ffor_without_belalp = compute_ffor(scenario_without_belalp)
ffor_with_belalp = compute_ffor(scenario_with_belalp)

In [ ]:
import matplotlib.pyplot as plt

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

plt.figure(figsize=(9, 8))
plt.plot(ffor_without_belalp["P"], ffor_without_belalp["Q"], "b-o", markersize=3, linewidth=1.5, label="Sans BelalpSolar")
plt.fill(ffor_without_belalp["P"], ffor_without_belalp["Q"], alpha=0.10, color="blue")
plt.plot(ffor_with_belalp["P"], ffor_with_belalp["Q"], "g-o", markersize=3, linewidth=1.5, label="Avec BelalpSolar")
plt.fill(ffor_with_belalp["P"], ffor_with_belalp["Q"], alpha=0.10, color="green")
plt.scatter([scenario_without_belalp["P_base"]], [scenario_without_belalp["Q_base"]], color="blue", marker="x", s=65, label="Point de base sans Belalp")
plt.scatter([scenario_with_belalp["P_base"]], [scenario_with_belalp["Q_base"]], color="green", marker="x", s=65, label="Point de base avec Belalp")
plt.xlabel("P_pcc (MW)")
plt.ylabel("Q_pcc (MVAr)")
plt.title("FFOR de la grille 87_0 sans et avec BelalpSolar")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.tight_layout()
plt.savefig(output_dir / "FFOR_without_and_with_belalp.png", dpi=150)
plt.show()

for scenario, ffor in [
    (scenario_without_belalp, ffor_without_belalp),
    (scenario_with_belalp, ffor_with_belalp),
]:
    print(f"{scenario['name']}: P_pcc=[{min(ffor['P']):.2f}, {max(ffor['P']):.2f}] MW, Q_pcc=[{min(ffor['Q']):.2f}, {max(ffor['Q']):.2f}] MVAr")

metadata_belalp = scenario_with_belalp["metadata"]
belalp_bus = int(metadata_belalp["belalp_bus"])
print("Metadonnees BelalpSolar:", metadata_belalp)
print(
    "Capacite reactive BelalpSolar dans le modele: "
    f"{-scenario_with_belalp['Q_pv_max'][belalp_bus]:.3f} a "
    f"{scenario_with_belalp['Q_pv_max'][belalp_bus]:.3f} MVAr | "
    f"S_inv={scenario_with_belalp['S_inv_max'][belalp_bus]:.3f} MVA | "
    f"cos(phi)={scenario_with_belalp['cos_phi']:.2f}"
)
